# Chapter 4 Project 2 - Mobile Robot Corridor Tracking

A unicycle robot tracks a circular reference inside an annular corridor. The MPC correction uses a local tracking-error model and anticipates a future narrowing corridor.

In [ ]:
%matplotlib inline
import os
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import numpy as np

from systems.mobile_robot import tracking_error_matrices, corridor_bounds
from mpc.terminal_tools import dare_terminal_cost
from mpc.casadi_linear_mpc import solve_linear_mpc
from scenarios.ch4_project2_mobile_robot import run_project2

cwd = Path.cwd()
repo_root = cwd if (cwd / "scenarios").exists() else cwd.parent
output_root = Path(os.environ.get("THIMPC_OUTPUT_ROOT", repo_root / "outputs"))

dt = 0.1
v_r = 0.8
omega_r = 0.2
A, B = tracking_error_matrices(dt, v_r, omega_r)
print("local tracking-error A =")
print(A)
print("local tracking-error B =")
print(B)


In [ ]:
# Baseline values keep the public notebook runnable.
Q = np.diag([1.0, 8.0, 1.0])
R = np.array([[1.0]])
horizon = 8
delta_bounds = (np.array([-0.7]), np.array([0.7]))
delta_rate = np.array([0.12])
soften_corridor = False
robust_margin = 0.0

# TODO: implement or tune this design choice.
# The surrounding setup is provided so you can focus on the control idea.

P_terminal = dare_terminal_cost(A, B, Q, R)
print("Q =")
print(Q)
print("R =")
print(R)
print("P_terminal =")
print(P_terminal)
print("yaw-rate correction bounds:", delta_bounds)
print("yaw-rate correction rate bound:", delta_rate)
print("soft corridor constraint:", soften_corridor)
print("robust margin for tightened example:", robust_margin)


## Visible Corridor-Constrained MPC Solve

The state is the local tracking error `[xi, eta, psi_error]`. The input is the yaw-rate correction added to the nominal circular-path yaw rate. Corridor bounds are visible as time-varying limits on lateral error `eta`.

In [ ]:
phi = omega_r * np.arange(horizon + 1) * dt
eta_min, eta_max = corridor_bounds(phi, margin=robust_margin if soften_corridor else 0.0)
lower = np.column_stack([np.full(horizon + 1, -2.0), eta_min, np.full(horizon + 1, -1.2)])
upper = np.column_stack([np.full(horizon + 1, 2.0), eta_max, np.full(horizon + 1, 1.2)])

x_error = np.array([0.0, 0.35, 0.12])
result = solve_linear_mpc(
    A,
    B,
    x_error,
    horizon,
    Q,
    R,
    P_terminal=P_terminal,
    u_bounds=delta_bounds,
    x_bounds_sequence=(lower, upper),
    rate_bound=delta_rate,
    u_previous=np.array([0.0]),
    soften_state_indices=[1] if soften_corridor else None,
    slack_penalty=20000.0,
)
print("first three eta lower bounds:", eta_min[:3])
print("first three eta upper bounds:", eta_max[:3])
print("solver success:", result.success)
print("solver status:", result.status)
print("first yaw-rate correction:", result.u0)
print("first predicted local error:", result.X[1])
if result.slack.size:
    print("initial corridor slack:", result.slack[0, 0])
else:
    print("no corridor slack variable used in this solve")


## Run the Full Application Study

In [ ]:
steps = int(os.environ.get("THIMPC_CH4_PROJECT2_STEPS", "120"))
metrics = run_project2(steps=steps, output_dir=output_root / "ch4_project2")
metrics
